# Music Genre Classification - Baseline

This time, we will only do basic stuff. We won't do anything fancy here.
This will serve as a baseline for us to compare with more advanced feature engineering techniques later.
Also, we want to try training multiple algorithms to see which models fit better with the music genre dataset.

## Preparation

In [1]:
from os.path import join
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Load dataset
path_dir = join("..", "..", "data")
df = pd.read_csv(join(path_dir, "preprocessed", "preprocessed_train.csv"))
df_test = pd.read_csv(join(path_dir, "preprocessed", "preprocessed_test.csv"))

df = df.drop(['Id', 'Artist Name', 'Track Name'], axis=1)
df_test = df_test.drop(['Id', 'Artist Name', 'Track Name'], axis=1)

df.to_csv(join(path_dir, "feature_engineered", "baseline", "baseline_train.csv"), index=False)
df_test.to_csv(join(path_dir, "feature_engineered", "baseline", "baseline_test.csv"), index=False)

df.head()


,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature,Class
0,37.0,0.334,-0.676218,9.0,0.153284,0,26.246030,0.723043,0.003912,-0.598627,0.235,1.014464,452.710724,4,9
1,67.0,0.725,0.256062,11.0,0.523145,1,11.415395,0.300739,0.045738,-0.629790,0.380,0.425560,438.127835,4,6
2,44.0,0.584,0.550185,7.0,0.332191,1,16.154828,0.098922,0.139563,1.055408,0.635,1.226456,401.294156,4,10
3,12.0,0.515,-1.433464,-1.0,-1.572682,1,32.050255,0.967986,0.021076,1.134827,0.501,1.563116,545.978937,3,2
4,48.0,0.565,0.408517,6.0,0.691321,0,4.016048,0.567741,0.003912,0.583192,0.619,-1.217968,504.127960,4,5


## Modelling

We tried several algorithms:
- Logistic Regression
- Ridge Classifier
- SGD Classifier
- Decision Tree
- Random Forest
- SVM
- KNN
- XGBoost
- LightGBM
- CatBoost

after that, we will choose 2 models that fits the dataset most to continue improve its accuracy.

In [2]:
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'Logistic': LogisticRegression(max_iter=1000),
    'RidgeCls': RidgeClassifier(),
    'SGD': SGDClassifier(random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42),
    'SVC': SVC(kernel='rbf', C=1.0),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'XGBoost': XGBClassifier(random_state=42, n_estimators=300, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, n_estimators=300),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='macro')
    rec = recall_score(y_val, y_pred, average='macro')
    f1 = f1_score(y_val, y_pred, average='macro')
    print(f"{name} -> acc:{acc:.4f} | prec:{prec:.4f} | rec:{rec:.4f} | f1:{f1:.4f}")

print("\nDetailed report (last model):")
print(classification_report(y_val, y_pred))

Logistic -> acc:0.4378 | prec:0.4247 | rec:0.4176 | f1:0.4020
RidgeCls -> acc:0.4406 | prec:0.4340 | rec:0.3425 | f1:0.3225
SGD -> acc:0.2656 | prec:0.3861 | rec:0.2056 | f1:0.1161
DecisionTree -> acc:0.3635 | prec:0.4477 | rec:0.4373 | f1:0.4416
RandomForest -> acc:0.5097 | prec:0.5689 | rec:0.5474 | f1:0.5519
SVC -> acc:0.3010 | prec:0.1920 | rec:0.1798 | f1:0.0815
KNN -> acc:0.3312 | prec:0.3565 | rec:0.3718 | f1:0.3619
XGBoost -> acc:0.4951 | prec:0.5782 | rec:0.5601 | f1:0.5665
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001377 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2662
[LightGBM] [Info] Number of data points in the train set: 11516, number of used features: 14
[LightGBM] [Info] Start training from score -3.360028
[LightGBM] [Info] Start t

Base on the outputs, we conclude:
- CatBoost: highest accuracy (0.5163) and highest macro recall (0.5634).
- XGBoost: highest macro F1 (0.5665) and precision (0.5782).

However, just based on local score is not enough, we need to compare Kaggle score to ensure our belief. We will take 4 best models (CatBoost, XGBoost, LightGBM, Random Forest), let them predict the genre of the music, and submit files on Kaggle to see it's actual accuracy. 

In [3]:
import os
import sys
import joblib
import numpy as np
from sklearn.model_selection import KFold

# === Thêm đường dẫn để import log_experiment ===
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
from log.experiment_logger import log_experiment

log_path = join(path_dir, '..', "log", 'baseLine', "experiment_log.csv")
author = "Thien"
models = {
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0),
    'XGBoost': XGBClassifier(random_state=42, n_estimators=300, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, n_estimators=300),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42),
}

In [ ]:
for name, model in models.items():
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    accuracy_scores = []
    precision_scores = []
    f1_scores = []
    recall_scores = []
    
    fold_index = 1
    for train_index, val_index in kf.split(X):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        acc = accuracy_score(y_val, y_pred)
        prec = precision_score(y_val, y_pred, average='macro')
        rec = recall_score(y_val, y_pred, average='macro')
        f1 = f1_score(y_val, y_pred, average='macro')
        
        accuracy_scores.append(acc)
        precision_scores.append(prec)
        f1_scores.append(f1)
        recall_scores.append(rec)
        
        print(f"\n==== Fold {fold_index} results for {name} ====")
        print(f"Fold {fold_index} -> acc:{acc:.4f} | prec:{prec:.4f} | rec:{rec:.4f} | f1:{f1:.4f}")
        fold_index += 1

    mean_acc = np.mean(accuracy_scores)
    mean_prec = np.mean(precision_scores)
    mean_f1 = np.mean(f1_scores)
    mean_rec = np.mean(recall_scores)
    print("\n==== Mean metrics ====")
    print(f"Acc: {mean_acc:.4f}")
    print(f"Prec: {mean_prec:.4f}")
    print(f"F1: {mean_f1:.4f}")
    print(f"Rec: {mean_rec:.4f}")

    # === Ghi log kết quả vào CSV ===
    log_experiment(
        output_path=log_path,
        model_name=name,
        feature_name="baseline",
        params= model.get_params(),
        kfold=5,
        acc=mean_acc,
        prec=mean_prec,
        f1=mean_f1,
        rec=mean_rec,
        author=author
    )

    # === Huấn luyện lại trên toàn bộ dữ liệu train ===
    final_model = model
    final_model.fit(X, y)

    # === Dump model ra .pkl ===
    model_dir = join(path_dir, '..', "log", "baseLine", "Model Pickles", name)
    os.makedirs(model_dir, exist_ok=True)
    model_path = join(model_dir, name + "_baseLine.pkl")
    joblib.dump(final_model, model_path)
    print(f"✅ Model saved to {model_path}")
    df_original = pd.read_csv(join(path_dir, "raw", "test.csv"))
    ids = df_original["Id"]
    # === Tạo file submission ===
    X_test = df_test.copy()
    if 'Class' in X_test.columns:
        X_test = X_test.drop(columns=['Class'])

    y_test_pred = final_model.predict(X_test)
    y_test_pred = y_test_pred.ravel()

    submission = pd.DataFrame({
        'Id': ids,  # đảm bảo test có cột này
        'Class': y_test_pred
    })

    sub_dir = join(path_dir, "submissions", "baseLine", name)
    os.makedirs(sub_dir, exist_ok=True)
    submission_path = join(sub_dir, f"submission_{name}_baseLine.csv")
    submission.to_csv(submission_path, index=False)
    print(f"📤 Submission file saved to {submission_path}")


==== Fold 1 results for CatBoost ====
Fold 1 -> acc:0.5260 | prec:0.5851 | rec:0.5789 | f1:0.5789

==== Fold 2 results for CatBoost ====
Fold 2 -> acc:0.5214 | prec:0.5663 | rec:0.5437 | f1:0.5488

==== Fold 3 results for CatBoost ====
Fold 3 -> acc:0.5304 | prec:0.5651 | rec:0.5726 | f1:0.5649

==== Fold 4 results for CatBoost ====
Fold 4 -> acc:0.5134 | prec:0.5675 | rec:0.5672 | f1:0.5647

==== Fold 5 results for CatBoost ====
Fold 5 -> acc:0.5196 | prec:0.5802 | rec:0.5707 | f1:0.5708

==== Mean metrics ====
Acc: 0.5222
Prec: 0.5728
F1: 0.5656
Rec: 0.5666
Logged experiment to ..\..\data\..\log\baseLine\experiment_log.csv
✅ Model saved to ..\..\data\..\log\baseLine\Model Pickles\CatBoost\CatBoost_baseLine.pkl
📤 Submission file saved to ..\..\data\submissions\baseLine\CatBoost\submission_CatBoost_baseLine.csv

==== Fold 1 results for XGBoost ====
Fold 1 -> acc:0.4927 | prec:0.5742 | rec:0.5534 | f1:0.5607

==== Fold 2 results for XGBoost ====
Fold 2 -> acc:0.4894 | prec:0.5462 | rec

## Result

<img src="images/output_kaggle_baseline.png" width="800">

This is what we received after submitting files on Kaggle platform. CatBoost is the best model we got, XGBoost is the second.
Further experiments is needed to improve model's accuracy.

# The end